<a href="https://colab.research.google.com/github/Racem1000/bess-optimizer/blob/main/notebooks/02_optimizer_phase2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"

folders = [
    "data/raw",          # raw fetched data (CSVs, JSON)
    "data/processed",    # cleaned Parquet files
    "models",            # trained ML models
    "results",           # backtest outputs, plots
    "notebooks",         # Colab notebook copies
]

for folder in folders:
    os.makedirs(f"{PROJECT_ROOT}/{folder}", exist_ok=True)

print(f"✅ Project structure created at: {PROJECT_ROOT}")
print("\nFolder tree:")
for folder in folders:
    print(f"  📁 {folder}")

✅ Project structure created at: /content/drive/MyDrive/bess-optimizer-sweden

Folder tree:
  📁 data/raw
  📁 data/processed
  📁 models
  📁 results
  📁 notebooks


In [ ]:
import pandas as pd
import numpy as np

# Create sample historical data (90 days)
# Note: 'h' is used instead of 'H' to avoid deprecation warnings
dates_hist = pd.date_range(end=pd.Timestamp.now(), periods=2160, freq='h')
df_hist = pd.DataFrame({
    'hour': dates_hist,
    'price': np.random.uniform(20, 100, size=len(dates_hist)),
    'zone': 'SE3'
})

# Create sample single day data
dates_hourly = pd.date_range(end=pd.Timestamp.now(), periods=24, freq='h')
df_hourly = pd.DataFrame({
    'hour': dates_hourly,
    'price': np.random.uniform(20, 100, size=len(dates_hourly)),
    'zone': 'SE3'
})

print(f"✅ Defined df_hist with {len(df_hist)} rows")
print(f"✅ Defined df_hourly with {len(df_hourly)} rows")

✅ Defined df_hist with 2160 rows
✅ Defined df_hourly with 24 rows


In [ ]:
# Save the data we've already pulled to persistent storage
df_hist.to_parquet(f"{PROJECT_ROOT}/data/processed/se3_hourly_90days_prototype.parquet")
df_hourly.to_parquet(f"{PROJECT_ROOT}/data/processed/se3_single_day_sample.parquet")

# Verify
import pandas as pd
loaded = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/se3_hourly_90days_prototype.parquet")
print(f"✅ Saved {len(loaded)} hourly rows of SE3 data to Drive")
print(f"   Date range: {loaded['hour'].min()} → {loaded['hour'].max()}")

✅ Saved 2160 hourly rows of SE3 data to Drive
   Date range: 2026-03-11 17:28:55.955193 → 2026-06-09 16:28:55.955193


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"
RAW_PATH = f"{PROJECT_ROOT}/data/raw"
PROCESSED_PATH = f"{PROJECT_ROOT}/data/processed"

def fetch_se3_day(date):
    """Fetch one day of SE3 prices. Returns list of dicts or None if unavailable."""
    url = f"https://www.elprisetjustnu.se/api/v1/prices/{date.year}/{date.strftime('%m-%d')}_SE3.json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

# Pull from elprisetjustnu API back to its earliest available date (~2022-11)
start_date = datetime(2022, 11, 1)
end_date = datetime.today() - timedelta(days=1)

print(f"Fetching SE3 prices from {start_date.date()} to {end_date.date()}")
print(f"Total days to fetch: {(end_date - start_date).days}")

all_data = []
current = start_date
days_missed = 0

with tqdm(total=(end_date - start_date).days) as pbar:
    while current <= end_date:
        data = fetch_se3_day(current)
        if data:
            all_data.extend(data)
        else:
            days_missed += 1
        current += timedelta(days=1)
        time.sleep(0.05)  # polite delay
        pbar.update(1)

print(f"\n✅ Fetched {len(all_data)} price points")
print(f"⚠️  Days with no data: {days_missed}")

# Save raw immediately so we don't lose this if something crashes downstream
raw_df = pd.DataFrame(all_data)
raw_df.to_parquet(f"{RAW_PATH}/se3_dayahead_raw.parquet")
print(f"\n💾 Raw saved to {RAW_PATH}/se3_dayahead_raw.parquet")

Fetching SE3 prices from 2022-11-01 to 2026-06-08
Total days to fetch: 1315


  1%|▏         | 19/1315 [00:04<05:18,  4.07it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

# Load the raw
raw = pd.read_parquet(f"{RAW_PATH}/se3_dayahead_raw.parquet")
print(f"Raw shape: {raw.shape}")
print(f"Raw columns: {raw.columns.tolist()}")
print(f"First row:\n{raw.iloc[0]}")

In [ ]:
import pandas as pd
import numpy as np

# 1. Parse timestamps to UTC (handles DST automatically because input has timezone)
df = raw.copy()
df['time_start'] = pd.to_datetime(df['time_start'], utc=True)
df['time_end']   = pd.to_datetime(df['time_end'], utc=True)

# 2. Compute duration to verify resolution per row
df['duration_min'] = (df['time_end'] - df['time_start']).dt.total_seconds() / 60
print("Resolution breakdown:")
print(df['duration_min'].value_counts())

# 3. Convert to industry-standard units (€/MWh and SEK/MWh)
df['price_sek_mwh'] = df['SEK_per_kWh'] * 1000
df['price_eur_mwh'] = df['EUR_per_kWh'] * 1000

# 4. Keep what we need, sort, deduplicate
df_clean = (
    df[['time_start', 'duration_min', 'price_sek_mwh', 'price_eur_mwh', 'EXR']]
    .rename(columns={'time_start': 'timestamp'})
    .sort_values('timestamp')
    .drop_duplicates(subset='timestamp')
    .set_index('timestamp')
)

# 5. Resample to hourly mean (aggregates 15-min data, leaves 60-min as-is)
hourly = df_clean[['price_sek_mwh', 'price_eur_mwh', 'EXR']].resample('1h').mean()
hourly = hourly.dropna(subset=['price_sek_mwh'])

# 6. Quality audit
expected_hours = int((hourly.index.max() - hourly.index.min()).total_seconds() / 3600) + 1
gap_count = expected_hours - len(hourly)
negative_count = (hourly['price_sek_mwh'] < 0).sum()
extreme_high = (hourly['price_sek_mwh'] > 5000).sum()

print(f"\n✅ Cleaned to {len(hourly)} hourly rows")
print(f"📅 Date range: {hourly.index.min()} → {hourly.index.max()}")
print(f"\nQuality report:")
print(f"  Expected hours:        {expected_hours:,}")
print(f"  Actual hours:          {len(hourly):,}")
print(f"  Gaps (missing hours):  {gap_count}")
print(f"  Negative price hours:  {negative_count} ({100*negative_count/len(hourly):.2f}%)")
print(f"  Extreme high (>5 SEK/kWh): {extreme_high}")

print(f"\nPrice statistics (SEK/MWh):")
print(hourly['price_sek_mwh'].describe().round(1))

print(f"\nPrice statistics (€/MWh):")
print(hourly['price_eur_mwh'].describe().round(1))

# 7. Save cleaned version
hourly.reset_index().to_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_hourly_2022_2026.parquet")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reload clean data
df = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
df = df.set_index("timestamp")

# Aggregate to daily
daily = df.resample("1D").agg({
    "price_eur_mwh": ["mean", "min", "max", "std"]
})
daily.columns = ["mean", "min", "max", "std"]
daily = daily.reset_index()

# Monthly negative price counts
df["is_negative"] = df["price_eur_mwh"] < 0
monthly_neg = df.resample("1M")["is_negative"].sum().reset_index()
monthly_neg.columns = ["month", "negative_hours"]

# 3-panel plot
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("Daily price evolution (€/MWh)",
                    "Daily price range (min/max bands)",
                    "Negative price hours per month"),
    vertical_spacing=0.08
)

# Panel 1: daily mean
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["mean"],
    name="Daily mean", line=dict(color="cyan", width=1)
), row=1, col=1)

# Panel 2: min-max band + mean
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["max"],
    line=dict(color="rgba(255,100,100,0.3)", width=0),
    showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["min"],
    fill="tonexty", fillcolor="rgba(255,255,255,0.15)",
    line=dict(color="rgba(100,100,255,0.3)", width=0),
    name="Daily range"
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["mean"],
    line=dict(color="yellow", width=1), name="Daily mean"
), row=2, col=1)

# Panel 3: negative hours per month
fig.add_trace(go.Bar(
    x=monthly_neg["month"], y=monthly_neg["negative_hours"],
    marker_color="crimson", name="Negative hours"
), row=3, col=1)

fig.update_layout(
    template="plotly_dark",
    height=800,
    title_text="SE3 Day-Ahead Prices — 4-Year History (Nov 2022 – Jun 2026)",
    showlegend=False
)
fig.update_yaxes(title_text="€/MWh", row=1, col=1)
fig.update_yaxes(title_text="€/MWh", row=2, col=1)
fig.update_yaxes(title_text="Hours", row=3, col=1)
fig.show()

# Print key insights
print(f"\n🔑 Key observations:")
print(f"  • 2022 energy crisis peak month: {daily.loc[daily['mean'].idxmax(), 'timestamp'].strftime('%b %Y')} "
      f"(mean €{daily['mean'].max():.0f}/MWh)")
print(f"  • Calmest month: {daily.loc[daily['std'].idxmin(), 'timestamp'].strftime('%b %Y')}")
print(f"  • Biggest single-day spread: €{(daily['max'] - daily['min']).max():.0f}/MWh "
      f"on {daily.loc[(daily['max']-daily['min']).idxmax(), 'timestamp'].strftime('%Y-%m-%d')}")
print(f"  • First month with >50 negative hours: "
      f"{monthly_neg[monthly_neg['negative_hours']>50].iloc[0]['month'].strftime('%b %Y') if (monthly_neg['negative_hours']>50).any() else 'never'}")

In [ ]:
import pandas as pd
import numpy as np
from pyomo.environ import *
from tqdm import tqdm

# Make sure Drive is mounted
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
except:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"
PROCESSED_PATH = f"{PROJECT_ROOT}/data/processed"

# 1. Load cleaned 4-year data from Drive (works even if session reset)
df = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
df["date"] = pd.to_datetime(df["timestamp"]).dt.date
print(f"✅ Loaded {len(df):,} hourly rows from Drive")

# 2. Define optimizer
def optimize_day_v2(prices_eur_mwh, capacity=2.0, max_power=1.0, efficiency=0.90, initial_soc=0.5):
    T = len(prices_eur_mwh)
    if T < 23 or T > 25:
        return None
    model = ConcreteModel()
    model.T = RangeSet(0, T-1)
    model.charge    = Var(model.T, bounds=(0, max_power))
    model.discharge = Var(model.T, bounds=(0, max_power))
    model.soc       = Var(model.T, bounds=(0, capacity))
    model.obj = Objective(
        expr=sum(prices_eur_mwh[t] * (model.discharge[t] - model.charge[t]) for t in model.T),
        sense=maximize
    )
    init_energy = initial_soc * capacity
    model.cons = ConstraintList()
    for t in model.T:
        if t == 0:
            model.cons.add(model.soc[t] == init_energy + efficiency*model.charge[t] - model.discharge[t])
        else:
            model.cons.add(model.soc[t] == model.soc[t-1] + efficiency*model.charge[t] - model.discharge[t])
    try:
        SolverFactory("appsi_highs").solve(model)
        discharge = [value(model.discharge[t]) for t in model.T]
        charge    = [value(model.charge[t])    for t in model.T]
        revenue   = sum(prices_eur_mwh[t] * (discharge[t] - charge[t]) for t in range(T))
        return {
            "revenue_eur": revenue,
            "cycles":      sum(discharge) / capacity,
            "spread":      max(prices_eur_mwh) - min(prices_eur_mwh),
            "mean_price":  float(np.mean(prices_eur_mwh)),
            "min_price":   float(min(prices_eur_mwh)),
            "max_price":   float(max(prices_eur_mwh)),
        }
    except Exception:
        return None

# 3. Run optimizer on every full day
print(f"\nBattery: 2 MWh / 1 MW / 90% RTE / starts at 50% SOC")
print(f"Running optimizer on every full day of SE3 history...\n")

results = []
for date, group in tqdm(df.groupby("date")):
    prices = group["price_eur_mwh"].values
    result = optimize_day_v2(prices)
    if result:
        result["date"] = date
        results.append(result)

results_df = pd.DataFrame(results)
results_df["date"] = pd.to_datetime(results_df["date"])
results_df.to_parquet(f"{PROCESSED_PATH}/se3_daily_arbitrage_results.parquet")

# 4. Print results
total_eur     = results_df["revenue_eur"].sum()
mean_daily    = results_df["revenue_eur"].mean()
median_daily  = results_df["revenue_eur"].median()
best_day      = results_df.loc[results_df["revenue_eur"].idxmax()]
worst_day     = results_df.loc[results_df["revenue_eur"].idxmin()]
total_cycles  = results_df["cycles"].sum()

print(f"\n{'='*60}")
print(f"📊 RESULTS — SE3 day-ahead arbitrage, Nov 2022 – Jun 2026")
print(f"{'='*60}")
print(f"Days optimized:        {len(results_df):,}")
print(f"Total cumulative:      €{total_eur:,.0f}")
print(f"Mean daily revenue:    €{mean_daily:.2f}")
print(f"Median daily revenue:  €{median_daily:.2f}")
print(f"Best day:              €{best_day['revenue_eur']:.0f}  on {best_day['date'].date()}")
print(f"Worst day:             €{worst_day['revenue_eur']:.0f}  on {worst_day['date'].date()}")
print(f"Total equivalent cycles: {total_cycles:.0f}")
print(f"Average cycles/year:   {total_cycles / (len(results_df)/365):.0f}")
print(f"\n💰 Annualized: €{mean_daily * 365:.0f}/year for the 2 MWh battery")
print(f"     → €{mean_daily * 365 / 1.0:.0f}/MW/year  (industry standard unit)")
print(f"     → €{mean_daily * 365 / 2.0:.0f}/MWh/year")

# 5. Yearly breakdown
results_df["year"] = results_df["date"].dt.year
yearly = results_df.groupby("year").agg(
    days=("revenue_eur", "count"),
    total_revenue=("revenue_eur", "sum"),
    mean_daily=("revenue_eur", "mean"),
    cycles=("cycles", "sum"),
).round(0)
print(f"\n📅 Yearly breakdown:")
print(yearly)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

results_df = pd.read_parquet(f"{PROCESSED_PATH}/se3_daily_arbitrage_results.parquet")
results_df["date"] = pd.to_datetime(results_df["date"])
results_df = results_df.sort_values("date").reset_index(drop=True)
results_df["revenue_30d_avg"] = results_df["revenue_eur"].rolling(30, min_periods=1).mean()
results_df["cumulative"] = results_df["revenue_eur"].cumsum()
results_df["year"] = results_df["date"].dt.year

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "Daily arbitrage revenue + 30-day rolling mean",
        "Cumulative revenue — the BESS wealth curve",
        "Revenue distribution by year — the duck-curve effect"
    ),
    vertical_spacing=0.10,
    row_heights=[0.35, 0.35, 0.30]
)

# Panel 1: daily scatter + rolling mean
fig.add_trace(go.Scatter(
    x=results_df["date"], y=results_df["revenue_eur"],
    mode="markers", marker=dict(size=3, color="cyan", opacity=0.35),
    name="Daily revenue"
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=results_df["date"], y=results_df["revenue_30d_avg"],
    mode="lines", line=dict(color="yellow", width=2.5),
    name="30-day rolling mean"
), row=1, col=1)

# Panel 2: cumulative
fig.add_trace(go.Scatter(
    x=results_df["date"], y=results_df["cumulative"],
    mode="lines", line=dict(color="lime", width=2),
    fill="tozeroy", fillcolor="rgba(0,255,0,0.15)",
    name="Cumulative", showlegend=False
), row=2, col=1)

# Panel 3: yearly box
for year in sorted(results_df["year"].unique()):
    year_data = results_df[results_df["year"] == year]
    fig.add_trace(go.Box(
        y=year_data["revenue_eur"], name=str(year),
        marker_color="orange", boxmean=True, showlegend=False
    ), row=3, col=1)

fig.update_layout(
    template="plotly_dark", height=900,
    title_text="BESS Arbitrage Revenue — SE3 4-Year Backtest (2 MWh / 1 MW / 90% RTE)"
)
fig.update_yaxes(title_text="€/day", row=1, col=1)
fig.update_yaxes(title_text="Cumulative €", row=2, col=1)
fig.update_yaxes(title_text="€/day", row=3, col=1)
fig.show()

In [ ]:
def optimize_day_deg(prices, capacity=2.0, max_power=1.0, efficiency=0.90,
                    initial_soc=0.5, c_deg=10.0):
    """LP with linear throughput degradation cost (€/MWh throughput)."""
    T = len(prices)
    if T < 23 or T > 25:
        return None
    m = ConcreteModel()
    m.T = RangeSet(0, T-1)
    m.charge    = Var(m.T, bounds=(0, max_power))
    m.discharge = Var(m.T, bounds=(0, max_power))
    m.soc       = Var(m.T, bounds=(0, capacity))
    m.obj = Objective(
        expr=sum(prices[t]*(m.discharge[t] - m.charge[t])
                 - c_deg*(m.charge[t] + m.discharge[t]) for t in m.T),
        sense=maximize
    )
    e0 = initial_soc * capacity
    m.cons = ConstraintList()
    for t in m.T:
        if t == 0:
            m.cons.add(m.soc[t] == e0 + efficiency*m.charge[t] - m.discharge[t])
        else:
            m.cons.add(m.soc[t] == m.soc[t-1] + efficiency*m.charge[t] - m.discharge[t])
    try:
        SolverFactory("appsi_highs").solve(m)
        dis = [value(m.discharge[t]) for t in m.T]
        chg = [value(m.charge[t]) for t in m.T]
        rev_gross = sum(prices[t]*(dis[t]-chg[t]) for t in range(T))
        thr = sum(dis) + sum(chg)
        return {
            "rev_gross": rev_gross,
            "deg_cost": c_deg * thr,
            "rev_net": rev_gross - c_deg * thr,
            "cycles": sum(dis) / capacity,
            "active": int(sum(dis) > 0.01),
        }
    except Exception:
        return None

# Sweep
deg_costs = [0, 5, 10, 20, 50]
sens = []

print("Running degradation sensitivity sweep...")
print(f"{'c_deg':>6} | {'€/year':>10} | {'cycles/yr':>10} | {'life(yr)':>9} | {'active days':>12}")
print("-" * 60)

for dc in deg_costs:
    daily = []
    for date, group in df.groupby("date"):
        prices = group["price_eur_mwh"].values
        r = optimize_day_deg(prices, c_deg=dc)
        if r:
            daily.append(r)

    d = pd.DataFrame(daily)
    years = len(d) / 365.25
    annual_net = d["rev_net"].sum() / years
    annual_cycles = d["cycles"].sum() / years
    battery_life = 6000 / annual_cycles if annual_cycles > 0 else 999
    active_days = d["active"].sum()

    sens.append({
        "c_deg": dc,
        "annual_revenue_net": annual_net,
        "annual_cycles": annual_cycles,
        "battery_life_years": battery_life,
        "active_days": active_days,
        "total_days": len(d),
    })
    print(f"€{dc:>4}/MWh | €{annual_net:>9,.0f} | {annual_cycles:>10.0f} | {battery_life:>9.1f} | {active_days:>5}/{len(d):>5}")

sens_df = pd.DataFrame(sens)
sens_df.to_parquet(f"{PROCESSED_PATH}/degradation_sensitivity.parquet")

# Pareto plot: revenue vs cycles
import plotly.express as px
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sens_df["annual_cycles"], y=sens_df["annual_revenue_net"],
    mode="lines+markers+text",
    text=[f"€{c}/MWh" for c in sens_df["c_deg"]],
    textposition="top center",
    line=dict(color="cyan", width=2),
    marker=dict(size=12, color="yellow")
))
fig.update_layout(
    template="plotly_dark",
    title="The Pareto Frontier — Revenue vs Cycles (SE3 4-year backtest)",
    xaxis_title="Annual equivalent full cycles",
    yaxis_title="Annual net revenue (€/MW/year)",
    height=500
)
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Load the Pareto results from the sweep
sens_df = pd.read_parquet(f"{PROCESSED_PATH}/degradation_sensitivity.parquet")

# Financial assumptions (industry standard for Swedish 2 MWh / 1 MW LFP)
CAPEX_PER_MWH       = 280_000   # €/MWh — installed cost, LFP, 2025
CAPACITY_MWH        = 2.0       # battery size
TOTAL_CAPEX         = CAPEX_PER_MWH * CAPACITY_MWH
WARRANTY_CYCLES     = 6000      # typical LFP warranty
WARRANTY_YEARS_MAX  = 15        # calendar life cap
DISCOUNT_RATE       = 0.08      # 8% — standard for energy project finance
FIXED_OM_PER_MW_YR  = 8000      # €/MW/year fixed O&M

# Compute NPV for each operating point
def compute_npv(annual_revenue, annual_cycles, capex=TOTAL_CAPEX,
                fixed_om=FIXED_OM_PER_MW_YR, r=DISCOUNT_RATE):
    # Battery life is min(warranty_cycles / cycles_per_year, calendar_max)
    life_cycles = WARRANTY_CYCLES / annual_cycles if annual_cycles > 0 else 999
    life_years  = min(life_cycles, WARRANTY_YEARS_MAX)

    # Annual net cashflow = revenue - O&M
    annual_cashflow = annual_revenue - fixed_om

    # NPV of cashflow stream over life_years
    npv_revenue = sum(annual_cashflow / (1 + r)**t for t in range(1, int(life_years) + 1))
    # Partial year at the end
    fractional = life_years - int(life_years)
    if fractional > 0:
        npv_revenue += (annual_cashflow * fractional) / (1 + r)**(int(life_years) + 1)

    # Total NPV = -CAPEX + NPV of cashflows
    npv = -capex + npv_revenue

    # IRR approximation (rough — solve annuity)
    payback_yrs = capex / annual_cashflow if annual_cashflow > 0 else 999

    return {
        "life_years": life_years,
        "annual_cashflow": annual_cashflow,
        "npv": npv,
        "payback_yrs": payback_yrs,
        "lifetime_revenue": annual_cashflow * life_years,
    }

# Apply to each operating point
sens_df["financials"] = sens_df.apply(
    lambda row: compute_npv(row["annual_revenue_net"], row["annual_cycles"]),
    axis=1
)
sens_df["life_years"]       = sens_df["financials"].apply(lambda x: x["life_years"])
sens_df["annual_cashflow"]  = sens_df["financials"].apply(lambda x: x["annual_cashflow"])
sens_df["npv"]              = sens_df["financials"].apply(lambda x: x["npv"])
sens_df["payback_yrs"]      = sens_df["financials"].apply(lambda x: x["payback_yrs"])
sens_df["lifetime_revenue"] = sens_df["financials"].apply(lambda x: x["lifetime_revenue"])

# Display
print(f"\n💰 NPV Analysis — 2 MWh / 1 MW LFP BESS in SE3")
print(f"   CAPEX: €{TOTAL_CAPEX:,.0f}  |  Discount: {DISCOUNT_RATE*100:.0f}%  |  Warranty: {WARRANTY_CYCLES} cycles, {WARRANTY_YEARS_MAX} yrs")
print(f"   Fixed O&M: €{FIXED_OM_PER_MW_YR}/MW/year")
print()
display_cols = ["c_deg", "annual_revenue_net", "annual_cycles", "life_years",
                "annual_cashflow", "lifetime_revenue", "npv", "payback_yrs"]
print(sens_df[display_cols].round(0).to_string(index=False))

# Pareto chart with NPV color
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sens_df["annual_cycles"],
    y=sens_df["annual_revenue_net"],
    mode="lines+markers+text",
    text=[f"€{c}/MWh<br>NPV: €{n:,.0f}<br>Life: {y:.1f}yr"
          for c, n, y in zip(sens_df["c_deg"], sens_df["npv"], sens_df["life_years"])],
    textposition="top center",
    line=dict(color="cyan", width=2),
    marker=dict(
        size=18,
        color=sens_df["npv"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="NPV (€)")
    )
))
fig.update_layout(
    template="plotly_dark",
    height=600,
    title=f"Pareto Frontier with NPV — Optimal Operating Point Selection<br><sub>CAPEX €{TOTAL_CAPEX:,.0f}, discount {DISCOUNT_RATE*100:.0f}%, warranty {WARRANTY_CYCLES} cycles / {WARRANTY_YEARS_MAX} yrs</sub>",
    xaxis_title="Annual equivalent cycles",
    yaxis_title="Annual net revenue (€/MW/year)",
)
fig.show()

# Identify the NPV-optimal point
best_idx = sens_df["npv"].idxmax()
best = sens_df.loc[best_idx]
print(f"\n🏆 NPV-optimal operating point:")
print(f"   Degradation cost: €{best['c_deg']:.0f}/MWh throughput")
print(f"   Annual revenue:   €{best['annual_revenue_net']:,.0f}/MW/year")
print(f"   Annual cycles:    {best['annual_cycles']:.0f}")
print(f"   Battery life:     {best['life_years']:.1f} years")
print(f"   Lifetime NPV:     €{best['npv']:,.0f}")
print(f"   Payback period:   {best['payback_yrs']:.1f} years")

In [ ]:
# At what CAPEX/MWh does DA-only arbitrage break even?
best_cashflow = sens_df.loc[sens_df["npv"].idxmax(), "annual_cashflow"]
best_life = sens_df.loc[sens_df["npv"].idxmax(), "life_years"]
r = 0.08

# PV of cashflow stream
pv_factor = sum(1 / (1 + r)**t for t in range(1, int(best_life) + 1))
breakeven_capex = best_cashflow * pv_factor

print(f"Best-case annual cashflow: €{best_cashflow:,.0f}")
print(f"Best-case battery life:    {best_life:.1f} years")
print(f"Discount factor sum:       {pv_factor:.2f}")
print(f"\n💰 Break-even total CAPEX: €{breakeven_capex:,.0f}")
print(f"   Break-even CAPEX/MWh:   €{breakeven_capex/2:,.0f}/MWh")
print(f"   Current CAPEX/MWh:      €280,000/MWh")
print(f"   Gap:                    €{280000 - breakeven_capex/2:,.0f}/MWh ({100*(280000 - breakeven_capex/2)/280000:.0f}% reduction needed)")
print(f"\nAt 10%/year CAPEX decline, that's {np.log(breakeven_capex / (560000)) / np.log(0.9):.1f} years away")

In [ ]:
import requests
import pandas as pd

lat, lon = 59.33, 18.06  # Stockholm/SE3

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2022-11-01",
    "end_date": "2026-06-04",
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "wind_speed_100m",
        "wind_direction_100m",
        "shortwave_radiation",
        "cloud_cover",
        "pressure_msl",
        "precipitation",        # mm/h - hydro inflow proxy
        "snowfall",             # cm/h - delayed inflow (snowmelt in spring)
    ]),
    "timezone": "UTC",
}

print("Fetching weather + precipitation from Open-Meteo...")
r = requests.get(url, params=params, timeout=60)
data = r.json()

weather_df = pd.DataFrame(data["hourly"])
weather_df["time"] = pd.to_datetime(weather_df["time"], utc=True)
weather_df = weather_df.rename(columns={"time": "timestamp"})

print(f"\n✅ Fetched {len(weather_df):,} hours")
print(f"📅 Range: {weather_df['timestamp'].min()} → {weather_df['timestamp'].max()}")
print(f"\nKey stats:")
print(f"  Temperature (°C):   {weather_df['temperature_2m'].min():.1f} → {weather_df['temperature_2m'].max():.1f}")
print(f"  Wind 100m (m/s):    mean {weather_df['wind_speed_100m'].mean():.1f}")
print(f"  Solar (W/m²):       mean {weather_df['shortwave_radiation'].mean():.0f}")
print(f"  Precipitation:      total {weather_df['precipitation'].sum():.0f} mm over period")

weather_df.to_parquet(f"{PROCESSED_PATH}/se3_weather_2022_2026.parquet")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_weather_2022_2026.parquet")

In [ ]:
import numpy as np

w = pd.read_parquet(f"{PROCESSED_PATH}/se3_weather_2022_2026.parquet")

# Aggregate to daily
daily_w = w.set_index("timestamp").resample("1D").agg({
    "temperature_2m": "mean",
    "wind_speed_100m": "mean",
    "shortwave_radiation": "mean",
    "precipitation": "sum",
    "snowfall": "sum",
}).reset_index()

# Hydro reservoir proxies — cumulative precipitation over various windows
# (Norwegian/Swedish reservoirs fill from rainfall and spring snowmelt with multi-week lag)
daily_w["precip_30d"]   = daily_w["precipitation"].rolling(30,  min_periods=1).sum()
daily_w["precip_90d"]   = daily_w["precipitation"].rolling(90,  min_periods=1).sum()
daily_w["precip_180d"]  = daily_w["precipitation"].rolling(180, min_periods=1).sum()
daily_w["snow_60d"]     = daily_w["snowfall"].rolling(60, min_periods=1).sum()

# "Reservoir proxy" — combined long-window precipitation
# High value = reservoirs filling = lower future prices expected (hydro abundance)
daily_w["reservoir_proxy"] = (
    daily_w["precip_180d"] / daily_w["precip_180d"].rolling(365, min_periods=30).mean()
)

daily_w.to_parquet(f"{PROCESSED_PATH}/se3_daily_weather_hydro.parquet")

print("✅ Hydro proxy features computed:")
print(f"   30-day precip:   mean {daily_w['precip_30d'].mean():.0f} mm  (range {daily_w['precip_30d'].min():.0f} - {daily_w['precip_30d'].max():.0f})")
print(f"   180-day precip:  mean {daily_w['precip_180d'].mean():.0f} mm")
print(f"   Reservoir proxy: range {daily_w['reservoir_proxy'].min():.2f} - {daily_w['reservoir_proxy'].max():.2f}  (1.0 = long-term average)")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_daily_weather_hydro.parquet")
print(f"   ({len(daily_w)} daily rows × {len(daily_w.columns)} columns)")

print("\n📝 Note: This is a precipitation-based proxy. For Phase 5 we may replace with")
print("   actual Nordic reservoir data from Nord Pool (manual download) or ENTSO-E.")

In [ ]:
df_prices = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")

# Day-ahead market mechanics (Nord Pool):
#   - Gate closure: 12:00 CET on D-1
#   - Market clearing + publication: ~13:00 CET on D-1
#   - Delivery: hours of day D
#
# So for any delivery_time on day D, the price became "known" at ~12:00 UTC on D-1
# (assuming CET = UTC+1, ignoring DST for simplicity; the timestamp is already UTC)

df_prices["delivery_date"] = pd.to_datetime(df_prices["timestamp"]).dt.date
df_prices["knowledge_time"] = (
    pd.to_datetime(df_prices["delivery_date"]) - pd.Timedelta(days=1)
).dt.tz_localize("UTC") + pd.Timedelta(hours=12)

# Save augmented version
df_prices.to_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026_v2.parquet")

# Verify
sample = df_prices.iloc[[0, 100, 1000, 10000, -1]]
print("✅ knowledge_time column added")
print(f"\nSample rows showing delivery_time vs knowledge_time:")
print(sample[["timestamp", "knowledge_time", "price_eur_mwh"]].to_string(index=False))

# Compute the gate-to-delivery lead time
lead = (pd.to_datetime(df_prices["timestamp"]) - df_prices["knowledge_time"])
print(f"\nLead time (delivery − gate publication):")
print(f"  Min:    {lead.min()}")
print(f"  Max:    {lead.max()}")
print(f"  Mean:   {lead.mean()}")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_hourly_2022_2026_v2.parquet")

print("\n📌 Why this matters: when we run rolling-horizon MPC in Phase 5, we'll filter")
print("   the dataset by 'knowledge_time <= decision_time' to prevent lookahead bias.")
print("   This methodology discipline is what separates 'student backtest' from 'research-grade'.")

In [ ]:
from pyomo.environ import *
import numpy as np
import pandas as pd
from tqdm import tqdm

def optimize_dispatch(
    prices,
    capacity_mwh=2.0,
    max_power_mw=1.0,
    efficiency=0.90,
    initial_soc_frac=0.5,
    final_soc_frac=None,
    degradation_cost=10.0,
    min_soc_frac=0.10,
    max_soc_frac=0.90,
    enforce_binary=True,
):
    """Production BESS dispatch optimizer (MILP or LP relaxation)."""
    prices = np.asarray(prices, dtype=float)
    T = len(prices)
    if T < 1:
        return None

    m = ConcreteModel()
    m.T = RangeSet(0, T-1)
    m.charge    = Var(m.T, bounds=(0, max_power_mw))
    m.discharge = Var(m.T, bounds=(0, max_power_mw))
    m.soc       = Var(m.T, bounds=(min_soc_frac * capacity_mwh, max_soc_frac * capacity_mwh))

    if enforce_binary:
        m.u_ch  = Var(m.T, within=Binary)
        m.u_dis = Var(m.T, within=Binary)

    m.obj = Objective(
        expr=sum(
            prices[t] * (m.discharge[t] - m.charge[t])
            - degradation_cost * (m.charge[t] + m.discharge[t])
            for t in m.T
        ),
        sense=maximize,
    )

    m.cons = ConstraintList()
    init_energy = initial_soc_frac * capacity_mwh

    for t in m.T:
        if t == 0:
            m.cons.add(m.soc[t] == init_energy + efficiency*m.charge[t] - m.discharge[t])
        else:
            m.cons.add(m.soc[t] == m.soc[t-1] + efficiency*m.charge[t] - m.discharge[t])
        if enforce_binary:
            m.cons.add(m.u_ch[t] + m.u_dis[t] <= 1)
            m.cons.add(m.charge[t]    <= max_power_mw * m.u_ch[t])
            m.cons.add(m.discharge[t] <= max_power_mw * m.u_dis[t])

    if final_soc_frac is not None:
        m.cons.add(m.soc[T-1] == final_soc_frac * capacity_mwh)

    result = SolverFactory("appsi_highs").solve(m)
    status = str(result.solver.termination_condition)

    try:
        charge_schedule    = np.array([value(m.charge[t])    for t in m.T])
        discharge_schedule = np.array([value(m.discharge[t]) for t in m.T])
        soc_schedule       = np.array([value(m.soc[t])       for t in m.T])
        revenue_gross = float(np.sum(prices * (discharge_schedule - charge_schedule)))
        throughput    = float(np.sum(charge_schedule + discharge_schedule))
        deg_cost_eur  = degradation_cost * throughput
        revenue_net   = revenue_gross - deg_cost_eur

        # Detect simultaneous charge/discharge (LP artifact)
        simultaneous = float(np.sum((charge_schedule > 0.001) & (discharge_schedule > 0.001)))

        return {
            "revenue_gross": revenue_gross,
            "degradation_cost_eur": deg_cost_eur,
            "revenue_net": revenue_net,
            "cycles": float(np.sum(discharge_schedule) / capacity_mwh),
            "throughput": throughput,
            "charge_schedule": charge_schedule,
            "discharge_schedule": discharge_schedule,
            "soc_schedule": soc_schedule,
            "simultaneous_hours": simultaneous,
            "status": status,
            "binary": enforce_binary,
        }
    except Exception as e:
        return {"status": f"failed: {e}", "binary": enforce_binary}


print("✅ optimize_dispatch defined.")
print("   Signature: optimize_dispatch(prices, capacity_mwh=2.0, max_power_mw=1.0, ...)")
print("   MILP mode: enforce_binary=True (default, proper)")
print("   LP relaxation: enforce_binary=False (faster, may have simultaneous dispatch artifacts)")

In [ ]:
# Load price data
df = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
df["date"] = pd.to_datetime(df["timestamp"]).dt.date

# Pick the best day from earlier backtest: 2024-12-12
sample_date = pd.to_datetime("2024-12-12").date()
sample_prices = df[df["date"] == sample_date]["price_eur_mwh"].values

print(f"📅 Sample day: {sample_date}")
print(f"   Hours: {len(sample_prices)}")
print(f"   Min/max price: €{sample_prices.min():.2f} / €{sample_prices.max():.2f}/MWh")
print(f"   Spread: €{sample_prices.max() - sample_prices.min():.2f}/MWh\n")

# Run LP relaxation
print("🔹 LP relaxation (enforce_binary=False):")
lp_result = optimize_dispatch(sample_prices, degradation_cost=10.0, enforce_binary=False)
print(f"   Revenue (gross): €{lp_result['revenue_gross']:.2f}")
print(f"   Degradation:     €{lp_result['degradation_cost_eur']:.2f}")
print(f"   Revenue (net):   €{lp_result['revenue_net']:.2f}")
print(f"   Cycles:          {lp_result['cycles']:.3f}")
print(f"   Simultaneous charge+discharge hours: {lp_result['simultaneous_hours']:.0f}")
print(f"   Status:          {lp_result['status']}")

# Run MILP
print("\n🔸 MILP (enforce_binary=True):")
milp_result = optimize_dispatch(sample_prices, degradation_cost=10.0, enforce_binary=True)
print(f"   Revenue (gross): €{milp_result['revenue_gross']:.2f}")
print(f"   Degradation:     €{milp_result['degradation_cost_eur']:.2f}")
print(f"   Revenue (net):   €{milp_result['revenue_net']:.2f}")
print(f"   Cycles:          {milp_result['cycles']:.3f}")
print(f"   Simultaneous charge+discharge hours: {milp_result['simultaneous_hours']:.0f}")
print(f"   Status:          {milp_result['status']}")

# Comparison
diff = milp_result['revenue_net'] - lp_result['revenue_net']
print(f"\n📊 MILP − LP difference: €{diff:.2f}")
if abs(diff) < 0.01:
    print("   ✅ Identical (LP found the optimal binary solution naturally)")
else:
    print(f"   ⚠️  Difference present — MILP enforces feasibility LP relaxation violated")

In [ ]:
print("Running MILP optimizer on every full day (1,311 days)...\n")

milp_results = []
for date, group in tqdm(df.groupby("date")):
    prices = group["price_eur_mwh"].values
    if len(prices) < 23 or len(prices) > 25:
        continue
    r = optimize_dispatch(
        prices,
        capacity_mwh=2.0,
        max_power_mw=1.0,
        efficiency=0.90,
        initial_soc_frac=0.5,
        degradation_cost=10.0,
        enforce_binary=True,
    )
    if r and "revenue_net" in r:
        milp_results.append({
            "date": date,
            "revenue_gross": r["revenue_gross"],
            "degradation_cost_eur": r["degradation_cost_eur"],
            "revenue_net": r["revenue_net"],
            "cycles": r["cycles"],
            "throughput": r["throughput"],
            "simultaneous_hours": r["simultaneous_hours"],
            "status": r["status"],
        })

milp_df = pd.DataFrame(milp_results)
milp_df["date"] = pd.to_datetime(milp_df["date"])
milp_df.to_parquet(f"{PROCESSED_PATH}/se3_daily_milp_results.parquet")

# Compare to LP baseline at €10/MWh (from earlier Pareto sweep)
sens = pd.read_parquet(f"{PROCESSED_PATH}/degradation_sensitivity.parquet")
lp_row = sens[sens["c_deg"] == 10].iloc[0]

milp_years = len(milp_df) / 365.25
milp_annual_net    = milp_df["revenue_net"].sum() / milp_years
milp_annual_cycles = milp_df["cycles"].sum() / milp_years
milp_total_simul   = milp_df["simultaneous_hours"].sum()
milp_active_days   = (milp_df["cycles"] > 0.01).sum()

print(f"\n{'='*70}")
print(f"📊 MILP vs LP comparison at €10/MWh degradation cost — 4-year backtest")
print(f"{'='*70}")
print(f"{'':30s} {'LP relaxation':>18s} {'MILP (binary)':>18s}")
print(f"{'-'*70}")
print(f"{'Annual revenue (€/MW/year)':30s} €{lp_row['annual_revenue_net']:>16,.0f}  €{milp_annual_net:>16,.0f}")
print(f"{'Annual equivalent cycles':30s} {lp_row['annual_cycles']:>18.0f}  {milp_annual_cycles:>18.0f}")
print(f"{'Active days':30s} {lp_row['active_days']:>18.0f}  {milp_active_days:>18.0f}")
print(f"{'Simul. charge+dis hours':30s} {'N/A':>18s}  {milp_total_simul:>18.0f}")
print(f"{'-'*70}")

diff = milp_annual_net - lp_row["annual_revenue_net"]
diff_pct = 100 * diff / lp_row["annual_revenue_net"]
print(f"\nRevenue Δ:  €{diff:+,.0f}/year  ({diff_pct:+.2f}%)")

if milp_total_simul == 0:
    print("✅ MILP confirms zero simultaneous charge+discharge — LP relaxation was already optimal at €10/MWh deg")
else:
    print(f"⚠️  MILP eliminated {milp_total_simul:.0f} simultaneous-dispatch hours that LP relaxation permitted")

# Yearly breakdown
milp_df["year"] = milp_df["date"].dt.year
yearly = milp_df.groupby("year").agg(
    days=("revenue_net", "count"),
    revenue_net=("revenue_net", "sum"),
    cycles=("cycles", "sum"),
).round(0)
print(f"\n📅 MILP yearly breakdown:")
print(yearly)

print(f"\n💾 Saved to {PROCESSED_PATH}/se3_daily_milp_results.parquet")

In [ ]:
soc_bands = [
    (0.0, 1.0,   "Aggressive (0-100%)"),
    (0.05, 0.95, "Standard (5-95%)"),
    (0.10, 0.90, "Conservative (10-90%)"),
    (0.15, 0.85, "Very conservative (15-85%)"),  # your choice
    (0.20, 0.80, "Extreme conservative (20-80%)"),
]

soc_sweep = []
for soc_min, soc_max, label in soc_bands:
    print(f"Running {label}...")
    daily = []
    for date, group in df.groupby("date"):
        prices = group["price_eur_mwh"].values
        if len(prices) < 23 or len(prices) > 25:
            continue
        r = optimize_dispatch(
            prices,
            capacity_mwh=2.0,
            max_power_mw=1.0,
            efficiency=0.90,
            degradation_cost=10.0,
            min_soc_frac=soc_min,
            max_soc_frac=soc_max,
            enforce_binary=True,
        )
        if r and "revenue_net" in r:
            daily.append(r["revenue_net"])

    annual = sum(daily) / (len(daily) / 365.25)
    usable_pct = (soc_max - soc_min) * 100
    soc_sweep.append({
        "label": label,
        "soc_min": soc_min,
        "soc_max": soc_max,
        "usable_pct": usable_pct,
        "annual_revenue": annual,
    })
    print(f"  → €{annual:,.0f}/MW/year  ({usable_pct:.0f}% usable)")

soc_sweep_df = pd.DataFrame(soc_sweep)
soc_sweep_df.to_parquet(f"{PROCESSED_PATH}/se3_soc_sensitivity.parquet")

print("\n📊 SOC range sensitivity:")
print(soc_sweep_df[["label", "usable_pct", "annual_revenue"]].to_string(index=False))

# Quick chart
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=soc_sweep_df["usable_pct"], y=soc_sweep_df["annual_revenue"],
    mode="lines+markers+text",
    text=soc_sweep_df["label"], textposition="bottom right",
    line=dict(color="cyan", width=2), marker=dict(size=12, color="yellow")
))
fig.update_layout(
    template="plotly_dark", height=500,
    title="SOC Range Sensitivity — Usable Capacity vs Annual Revenue",
    xaxis_title="Usable capacity (% of nominal)",
    yaxis_title="Annual net revenue (€/MW/year)"
)
fig.show()

In [ ]:
# Re-run MILP at 10-90% SOC as the production baseline
print("Running production MILP at 10-90% SOC baseline (1,311 days)...\n")

baseline_results = []
for date, group in tqdm(df.groupby("date")):
    prices = group["price_eur_mwh"].values
    if len(prices) < 23 or len(prices) > 25:
        continue
    r = optimize_dispatch(
        prices,
        capacity_mwh=2.0,
        max_power_mw=1.0,
        efficiency=0.90,
        degradation_cost=10.0,
        min_soc_frac=0.10,      # ← updated
        max_soc_frac=0.90,      # ← updated
        enforce_binary=True,
    )
    if r and "revenue_net" in r:
        baseline_results.append({
            "date": date,
            "revenue_net": r["revenue_net"],
            "revenue_gross": r["revenue_gross"],
            "cycles": r["cycles"],
            "throughput": r["throughput"],
        })

baseline_df = pd.DataFrame(baseline_results)
baseline_df["date"] = pd.to_datetime(baseline_df["date"])
baseline_df.to_parquet(f"{PROCESSED_PATH}/se3_milp_baseline_10_90.parquet")

# Aggregate
years = len(baseline_df) / 365.25
annual_revenue = baseline_df["revenue_net"].sum() / years
annual_cycles  = baseline_df["cycles"].sum() / years

# NPV
CAPEX_TOTAL     = 560_000   # €
OM_PER_MW_YR    = 8_000
DISCOUNT_RATE   = 0.08
WARRANTY_CYCLES = 6_000
WARRANTY_YEARS  = 15

life_years      = min(WARRANTY_CYCLES / annual_cycles, WARRANTY_YEARS)
annual_cashflow = annual_revenue - OM_PER_MW_YR
discount_sum    = sum(1/(1 + DISCOUNT_RATE)**t for t in range(1, int(life_years) + 1))
pv_cashflow     = annual_cashflow * discount_sum
npv             = -CAPEX_TOTAL + pv_cashflow

print(f"\n{'='*65}")
print(f"💰 PRODUCTION BASELINE — MILP at 10-90% SOC")
print(f"{'='*65}")
print(f"Annual revenue:           €{annual_revenue:>10,.0f}/MW/year")
print(f"Annual cycles:            {annual_cycles:>11.0f}")
print(f"Battery life:             {life_years:>11.1f} years")
print(f"Annual cashflow (after O&M): €{annual_cashflow:>7,.0f}/MW/year")
print(f"Discount factor sum:      {discount_sum:>11.2f}")
print(f"PV of cashflow:           €{pv_cashflow:>10,.0f}")
print(f"CAPEX:                    €{CAPEX_TOTAL:>10,.0f}")
print(f"NPV:                      €{npv:>10,.0f}")

# Phase 4 gap
required_increase = -npv / discount_sum
print(f"\n{'='*65}")
print(f"🎯 PHASE 4 TARGET — what FCR-N + FCR-D must deliver")
print(f"{'='*65}")
print(f"NPV gap to close:                €{-npv:,.0f}")
print(f"Required additional revenue:     €{required_increase:,.0f}/MW/year")
print(f"As % of current arbitrage rev:   {100*required_increase/annual_revenue:.0f}%")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_milp_baseline_10_90.parquet")

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Load the production baseline results (10-90% MILP)
baseline_df = pd.read_parquet(f"{PROCESSED_PATH}/se3_milp_baseline_10_90.parquet")
baseline_df["date"] = pd.to_datetime(baseline_df["date"])

# Baseline (already computed): €10/MWh effective cost (degradation only)
# Transaction costs are REAL market fees — added on top of degradation
# Nord Pool DA fee: ~€0.05/MWh, Swedish grid tariff for storage: ~€1.20/MWh,
# BRP imbalance buffer: ~€0.25/MWh → total ~€1.50/MWh

tx_scenarios = [
    (0.0,  "No transaction costs (current baseline)"),
    (0.5,  "Minimal fees only (€0.50/MWh)"),
    (1.5,  "Standard SE3 fees (€1.50/MWh)"),
    (3.0,  "Conservative estimate (€3.00/MWh)"),
]

print("Transaction cost sensitivity...\n")
tx_results = []
df_prices = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
df_prices["date"] = pd.to_datetime(df_prices["timestamp"]).dt.date

for tc, label in tx_scenarios:
    effective_cost = 10.0 + tc  # deg + tx
    daily = []
    for date, group in df_prices.groupby("date"):
        prices = group["price_eur_mwh"].values
        if len(prices) < 23 or len(prices) > 25:
            continue
        r = optimize_dispatch(
            prices,
            degradation_cost=effective_cost,
            min_soc_frac=0.10,
            max_soc_frac=0.90,
            enforce_binary=True,
        )
        if r and "revenue_net" in r:
            daily.append(r["revenue_net"])

    annual = sum(daily) / (len(daily) / 365.25)
    tx_results.append({"tx_cost": tc, "label": label, "annual_revenue": annual})
    print(f"  €{tc:.1f}/MWh: €{annual:,.0f}/MW/year")

tx_df = pd.DataFrame(tx_results)
baseline_rev = tx_df[tx_df["tx_cost"] == 0]["annual_revenue"].values[0]
standard_rev = tx_df[tx_df["tx_cost"] == 1.5]["annual_revenue"].values[0]

print(f"\n📊 Impact of standard SE3 transaction costs (€1.50/MWh):")
print(f"   Before: €{baseline_rev:,.0f}/MW/year")
print(f"   After:  €{standard_rev:,.0f}/MW/year")
print(f"   Loss:   €{baseline_rev - standard_rev:,.0f}/MW/year ({100*(baseline_rev - standard_rev)/baseline_rev:.1f}%)")
tx_df.to_parquet(f"{PROCESSED_PATH}/se3_transaction_cost_sensitivity.parquet")

In [ ]:
# Define market regimes from our earlier market analysis
# These boundaries match what we observed in the price evolution chart
REGIMES = {
    "crisis":               ("2022-11-01", "2023-03-31"),
    "normalization":        ("2023-04-01", "2024-09-30"),
    "renewable_saturation": ("2024-10-01", "2026-06-04"),
}

def assign_regime(date):
    d = pd.Timestamp(date)
    if d <= pd.Timestamp("2023-03-31"):
        return "crisis"
    elif d <= pd.Timestamp("2024-09-30"):
        return "normalization"
    else:
        return "renewable_saturation"

baseline_df["regime"] = baseline_df["date"].apply(lambda x: assign_regime(x))

print("Regime-aware backtest split (production MILP, 10-90% SOC, €10/MWh deg):\n")
print(f"{'Regime':<25} {'Days':>6} {'Years':>6} {'€/day':>8} {'€/MW/yr':>10} {'Cycles/yr':>10} {'Neg Price %':>12}")
print("─" * 80)

regime_results = {}
for regime in ["crisis", "normalization", "renewable_saturation"]:
    g = baseline_df[baseline_df["regime"] == regime]
    if len(g) == 0:
        continue

    years     = len(g) / 365.25
    daily_rev = g["revenue_net"].mean()
    annual    = g["revenue_net"].sum() / years
    cycles    = g["cycles"].sum() / years

    # Negative price % from price data
    prices_reg = df_prices[df_prices["date"].apply(assign_regime) == regime]
    neg_pct = 100 * (prices_reg["price_eur_mwh"] < 0).mean()

    regime_results[regime] = {
        "days": len(g), "years": years, "daily_rev": daily_rev,
        "annual_rev": annual, "cycles": cycles, "neg_pct": neg_pct
    }

    print(f"{regime:<25} {len(g):>6} {years:>6.2f} €{daily_rev:>7.0f} €{annual:>9,.0f} {cycles:>10.0f}  {neg_pct:>11.1f}%")

print("─" * 80)
years_total = len(baseline_df) / 365.25
print(f"{'ALL REGIMES':<25} {len(baseline_df):>6} {years_total:>6.2f} €{baseline_df['revenue_net'].mean():>7.0f} €{baseline_df['revenue_net'].sum()/years_total:>9,.0f} {baseline_df['cycles'].sum()/years_total:>10.0f}")

print("\n📌 Key insight: does arbitrage value grow or shrink over time?")
crisis_rev = regime_results["crisis"]["annual_rev"]
sat_rev = regime_results["renewable_saturation"]["annual_rev"]
print(f"   Crisis → Renewable saturation: €{crisis_rev:,.0f} → €{sat_rev:,.0f}")
print(f"   Change: {100*(sat_rev - crisis_rev)/crisis_rev:+.0f}% {'↑ growing' if sat_rev > crisis_rev else '↓ shrinking'}")

In [ ]:
import requests
import pandas as pd

url = "3eBak6-z42UgQAGHxf3BFlackQJ8dsv1"

response = requests.get(url)
data = response.json()

df = pd.DataFrame(data)

In [ ]:
from google.colab import files
uploaded = files.upload()  # select all 4 FCR CSVs

Saving FCR_4.csv to FCR_4.csv
Saving FCR_3.csv to FCR_3.csv
Saving FCR_2.csv to FCR_2.csv
Saving FCR_(1).csv to FCR_(1).csv


In [ ]:
FCR_FILES = [
    "/content/FCR_(1).csv",
    "/content/FCR_2.csv",
    "/content/FCR_3.csv",
    "/content/FCR_4.csv",
]

In [ ]:
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"
PROCESSED_PATH = f"{PROJECT_ROOT}/data/processed"

def load_fcr_prices(file_paths: list[str]) -> pd.DataFrame:
    """
    Load and merge real FCR price data exported from Svenska kraftnät.
    Columns: timestamp, fcr_n, fcr_d_up, fcr_d_dn  (€/MW/h)
    Also retains SE3-specific volume columns for optional capacity weighting.
    """
    dfs = []
    for path in file_paths:
        df = pd.read_csv(
            path,
            sep=";",
            encoding="utf-8-sig",
            on_bad_lines="skip",
            decimal=",",
        )
        # Parse and localize to UTC to match the spot price dataframe
        df["timestamp"] = pd.to_datetime(df["Datum"], errors="coerce").dt.tz_localize("UTC")
        dfs.append(df)

    raw = (
        pd.concat(dfs, ignore_index=True)
        .sort_values("timestamp")
        .drop_duplicates("timestamp")
        .reset_index(drop=True)
    )

    return raw.rename(columns={
        "FCR-N Pris (EUR/MW)":     "fcr_n",
        "FCR-D upp Pris (EUR/MW)": "fcr_d_up",
        "FCR-D ned Pris (EUR/MW)": "fcr_d_dn",
        "SE3 FCRN":                "se3_fcrn_vol",
        "SE3 FCRD upp":            "se3_fcrd_up_vol",
        "SE3 FCRD ned":            "se3_fcrd_dn_vol",
    })[["timestamp", "fcr_n", "fcr_d_up", "fcr_d_dn",
        "se3_fcrn_vol", "se3_fcrd_up_vol", "se3_fcrd_dn_vol"]]


# ── Build Phase 4 dataset ────────────────────────────────────────────────────
df_prices = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026_v2.parquet")
df_prices["date"] = pd.to_datetime(df_prices["timestamp"]).dt.date

fcr_all = load_fcr_prices(FCR_FILES)

# Merge real FCR prices onto the spot price dataframe
df_ph4 = (
    df_prices[
        (df_prices["timestamp"] >= "2024-01-01") &
        (df_prices["timestamp"] <  "2026-01-01")
    ]
    .copy()
    .merge(fcr_all[["timestamp", "fcr_n", "fcr_d_up", "fcr_d_dn"]],
           on="timestamp", how="left")
)

print(f"Phase 4 dataset: {len(df_ph4):,} hours "
      f"({df_ph4['timestamp'].min().date()} → {df_ph4['timestamp'].max().date()})")
print(f"\nFCR price summary (€/MW/h) — real SVK data:")
print(f"  FCR-N:    mean €{df_ph4['fcr_n'].mean():.2f}  | "
      f"range €{df_ph4['fcr_n'].min():.2f} – €{df_ph4['fcr_n'].max():.2f}")
print(f"  FCR-D up: mean €{df_ph4['fcr_d_up'].mean():.2f}  | "
      f"range €{df_ph4['fcr_d_up'].min():.2f} – €{df_ph4['fcr_d_up'].max():.2f}")
print(f"  FCR-D dn: mean €{df_ph4['fcr_d_dn'].mean():.2f}  | "
      f"range €{df_ph4['fcr_d_dn'].min():.2f} – €{df_ph4['fcr_d_dn'].max():.2f}")
print(f"\nAnnualized FCR-N at full reservation (1 MW):")
print(f"  €{df_ph4['fcr_n'].mean() * 8760:,.0f}/MW/year")

Phase 4 dataset: 17,520 hours (2024-01-01 → 2025-12-31)

FCR price summary (€/MW/h) — real SVK data:
  FCR-N:    mean €37.28  | range €7.63 – €658.38
  FCR-D up: mean €8.28  | range €0.76 – €336.73
  FCR-D dn: mean €16.47  | range €0.85 – €2197.72

Annualized FCR-N at full reservation (1 MW):
  €326,560/MW/year


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from pyomo.environ import *
import numpy as np

def optimize_dispatch_multimarket(
    da_prices,
    fcr_n_prices,
    fcr_d_up_prices=None,
    fcr_d_dn_prices=None,
    capacity_mwh=2.0,
    max_power_mw=1.0,
    efficiency=0.90,
    initial_soc_frac=0.5,
    degradation_cost=10.0,
    min_soc_frac=0.10,
    max_soc_frac=0.90,
    fcr_response_hours=0.25,
    alpha_fcr_n=0.05,
    alpha_fcr_d=0.03,
):
    da_prices = np.asarray(da_prices, dtype=float)
    fcr_n_prices = np.asarray(fcr_n_prices, dtype=float)
    T = len(da_prices)

    use_d_up = fcr_d_up_prices is not None
    use_d_dn = fcr_d_dn_prices is not None
    fcr_d_up = np.asarray(fcr_d_up_prices, dtype=float) if use_d_up else np.zeros(T)
    fcr_d_dn = np.asarray(fcr_d_dn_prices, dtype=float) if use_d_dn else np.zeros(T)

    m = ConcreteModel()
    m.T = RangeSet(0, T-1)
    m.charge = Var(m.T, bounds=(0, max_power_mw))
    m.discharge = Var(m.T, bounds=(0, max_power_mw))
    m.soc = Var(m.T, bounds=(min_soc_frac * capacity_mwh, max_soc_frac * capacity_mwh))
    m.u_ch = Var(m.T, within=Binary)
    m.u_dis = Var(m.T, within=Binary)
    m.r_fcr_n = Var(m.T, bounds=(0, max_power_mw))
    if use_d_up: m.r_d_up = Var(m.T, bounds=(0, max_power_mw))
    if use_d_dn: m.r_d_dn = Var(m.T, bounds=(0, max_power_mw))

    def obj_expr(m):
        total = 0
        for t in m.T:
            total += da_prices[t] * (m.discharge[t] - m.charge[t])
            total += fcr_n_prices[t] * m.r_fcr_n[t]
            if use_d_up: total += fcr_d_up[t] * m.r_d_up[t]
            if use_d_dn: total += fcr_d_dn[t] * m.r_d_dn[t]
            deg = (m.charge[t] + m.discharge[t] + alpha_fcr_n * m.r_fcr_n[t])
            if use_d_up: deg += alpha_fcr_d * m.r_d_up[t]
            if use_d_dn: deg += alpha_fcr_d * m.r_d_dn[t]
            total -= degradation_cost * deg
        return total

    m.obj = Objective(rule=obj_expr, sense=maximize)
    m.cons = ConstraintList()
    init_e = initial_soc_frac * capacity_mwh

    for t in m.T:
        if t == 0:
            m.cons.add(m.soc[t] == init_e + efficiency*m.charge[t] - m.discharge[t])
        else:
            m.cons.add(m.soc[t] == m.soc[t-1] + efficiency*m.charge[t] - m.discharge[t])
        m.cons.add(m.u_ch[t] + m.u_dis[t] <= 1)
        m.cons.add(m.charge[t] <= max_power_mw * m.u_ch[t])
        m.cons.add(m.discharge[t] <= max_power_mw * m.u_dis[t])
        ch_total = m.charge[t] + m.r_fcr_n[t] + (m.r_d_dn[t] if use_d_dn else 0)
        dis_total = m.discharge[t] + m.r_fcr_n[t] + (m.r_d_up[t] if use_d_up else 0)
        m.cons.add(ch_total <= max_power_mw)
        m.cons.add(dis_total <= max_power_mw)
        up_h = m.r_fcr_n[t] + (m.r_d_up[t] if use_d_up else 0)
        dn_h = m.r_fcr_n[t] + (m.r_d_dn[t] if use_d_dn else 0)
        m.cons.add(m.soc[t] >= min_soc_frac * capacity_mwh + up_h * fcr_response_hours)
        m.cons.add(m.soc[t] <= max_soc_frac * capacity_mwh - dn_h * fcr_response_hours)

    solver = SolverFactory("appsi_highs")
    try:
        result = solver.solve(m)
        status = str(result.solver.termination_condition)
        if status != "optimal":
            return {"status": status, "revenue_net": None}
        ch = np.array([value(m.charge[t]) for t in m.T])
        dis = np.array([value(m.discharge[t]) for t in m.T])
        fcr_n_s = np.array([value(m.r_fcr_n[t]) for t in m.T])
        d_up_s = np.array([value(m.r_d_up[t]) for t in m.T]) if use_d_up else np.zeros(T)
        d_dn_s = np.array([value(m.r_d_dn[t]) for t in m.T]) if use_d_dn else np.zeros(T)
        da_rev = float(np.sum(da_prices * (dis - ch)))
        fcr_n_rev = float(np.sum(fcr_n_prices * fcr_n_s))
        fcr_d_rev = float(np.sum(fcr_d_up * d_up_s + fcr_d_dn * d_dn_s))
        thr = float(np.sum(ch + dis) + alpha_fcr_n * np.sum(fcr_n_s) + alpha_fcr_d * (np.sum(d_up_s + d_dn_s)))
        rev_net = da_rev + fcr_n_rev + fcr_d_rev - (degradation_cost * thr)
        return {"da_revenue": da_rev, "fcr_n_revenue": fcr_n_rev, "fcr_d_revenue": fcr_d_rev, "revenue_net": rev_net, "cycles": float(np.sum(dis)/capacity_mwh), "fcr_n_utilization": float(np.mean(fcr_n_s)/max_power_mw), "status": "optimal"}
    except Exception as e:
        return {"status": f"error: {e}", "revenue_net": None}

In [ ]:
from pyomo.environ import *
import numpy as np
import pandas as pd

def optimize_dispatch(
    prices,
    capacity_mwh=2.0,
    max_power_mw=1.0,
    efficiency=0.90,
    initial_soc_frac=0.5,
    final_soc_frac=None,
    degradation_cost=10.0,
    min_soc_frac=0.10,
    max_soc_frac=0.90,
    enforce_binary=True,
):
    """Production BESS dispatch optimizer (MILP)."""
    prices = np.asarray(prices, dtype=float)
    T = len(prices)
    m = ConcreteModel()
    m.T = RangeSet(0, T-1)
    m.charge    = Var(m.T, bounds=(0, max_power_mw))
    m.discharge = Var(m.T, bounds=(0, max_power_mw))
    m.soc       = Var(m.T, bounds=(min_soc_frac * capacity_mwh, max_soc_frac * capacity_mwh))

    if enforce_binary:
        m.u_ch  = Var(m.T, within=Binary)
        m.u_dis = Var(m.T, within=Binary)

    m.obj = Objective(
        expr=sum(prices[t] * (m.discharge[t] - m.charge[t]) - degradation_cost * (m.charge[t] + m.discharge[t]) for t in m.T),
        sense=maximize
    )

    m.cons = ConstraintList()
    init_energy = initial_soc_frac * capacity_mwh
    for t in m.T:
        if t == 0:
            m.cons.add(m.soc[t] == init_energy + efficiency*m.charge[t] - m.discharge[t])
        else:
            m.cons.add(m.soc[t] == m.soc[t-1] + efficiency*m.charge[t] - m.discharge[t])
        if enforce_binary:
            m.cons.add(m.u_ch[t] + m.u_dis[t] <= 1)
            m.cons.add(m.charge[t]    <= max_power_mw * m.u_ch[t])
            m.cons.add(m.discharge[t] <= max_power_mw * m.u_dis[t])

    SolverFactory("appsi_highs").solve(m)
    dis = np.array([value(m.discharge[t]) for t in m.T])
    ch  = np.array([value(m.charge[t]) for t in m.T])
    rev_gross = float(np.sum(prices * (dis - ch)))
    deg_cost  = degradation_cost * float(np.sum(ch + dis))

    return {
        "revenue_net": rev_gross - deg_cost,
        "da_revenue": rev_gross,
        "cycles": float(np.sum(dis) / capacity_mwh),
        "status": "optimal"
    }

# Use 2024-12-12 — the best DA day — to validate multi-market behavior
sample_date = pd.to_datetime("2024-12-12").date()
sample = df_ph4[df_ph4["date"] == sample_date]

if len(sample) == 0:
    print("Date not in Phase 4 dataset. Using first available day.")
    sample = df_ph4[df_ph4["date"] == df_ph4["date"].unique()[0]]

da_p   = sample["price_eur_mwh"].values
fcr_n  = sample["fcr_n"].values
fcr_du = sample["fcr_d_up"].values
fcr_dd = sample["fcr_d_dn"].values

print(f"$📅 Sample day: {sample['date'].iloc[0]}")
print(f"   DA spread:     €{da_p.max()-da_p.min():.0f}/MWh")
print(f"   FCR-N avg:     €{fcr_n.mean():.2f}/MW/h → €{fcr_n.mean()*24:.0f}/day at 1 MW")
print(f"   FCR-D avg:     up €{fcr_du.mean():.2f} / dn €{fcr_dd.mean():.2f}")

# Run three scenarios
r_da      = optimize_dispatch(da_p, degradation_cost=10.0, min_soc_frac=0.10, max_soc_frac=0.90)
r_fcr_n   = optimize_dispatch_multimarket(da_p, fcr_n)
r_fcr_all = optimize_dispatch_multimarket(da_p, fcr_n, fcr_du, fcr_dd)

print(f"\n{'Scenario':<25} {'DA rev':>8} {'FCR-N':>8} {'FCR-D':>8} {'Net rev':>9} {'Cycles':>7} {'FCR-N%':>8}")
print("─" * 80)
for label, r in [("DA-only", r_da), ("DA + FCR-N", r_fcr_n), ("DA + FCR-N + FCR-D", r_fcr_all)]:
    if "revenue_net" not in r:
        print(f"{label:<25} SOLVER FAILED: {r.get('status')}")
        continue
    fcr_n_rev = r.get("fcr_n_revenue", 0)
    fcr_d_rev = r.get("fcr_d_revenue", 0)
    da_rev    = r.get("da_revenue", r.get("revenue_gross", 0))
    net       = r["revenue_net"]
    cycles    = r["cycles"]
    util      = r.get("fcr_n_utilization", 0)
    print(f"{label:<25} €{da_rev:>6.0f}  €{fcr_n_rev:>6.0f}  €{fcr_d_rev:>6.0f}  €{net:>7.0f}  {cycles:>6.2f}  {100*util:>6.1f}%")

$📅 Sample day: 2024-12-12
   DA spread:     €659/MWh
   FCR-N avg:     €16.42/MW/h → €394/day at 1 MW
   FCR-D avg:     up €6.11 / dn €2.44

Scenario                    DA rev    FCR-N    FCR-D   Net rev  Cycles   FCR-N%
────────────────────────────────────────────────────────────────────────────────
DA-only                   €  1193  €     0  €     0  €   1134    1.60     0.0%
DA + FCR-N                €  1148  €   228  €     0  €   1314    1.47    52.5%
DA + FCR-N + FCR-D        €  1165  €   211  €    49  €   1356    1.60    47.6%


In [ ]:
results = {"da_only": [], "da_fcr_n": [], "da_fcr_all": []}
infeasible_counts = {"da_only": 0, "da_fcr_n": 0, "da_fcr_all": 0}

for date, group in tqdm(df_ph4.groupby("date"), desc="Backtesting Days"):
    if len(group) < 23 or len(group) > 25: continue
    da_p = group["price_eur_mwh"].values
    fcr_n = group["fcr_n"].values
    fcr_du = group["fcr_d_up"].values
    fcr_dd = group["fcr_d_dn"].values

    # DA Only
    r = optimize_dispatch(da_p, degradation_cost=10.0)
    if r.get("revenue_net") is not None:
        results["da_only"].append({"date": date, "revenue_net": r["revenue_net"], "da_revenue": r["da_revenue"], "fcr_n_revenue": 0, "fcr_d_revenue": 0, "cycles": r["cycles"], "fcr_n_utilization": 0})

    # DA + FCR-N
    r = optimize_dispatch_multimarket(da_p, fcr_n)
    if r.get("status") == "optimal":
        results["da_fcr_n"].append({"date": date, "revenue_net": r["revenue_net"], "da_revenue": r["da_revenue"], "fcr_n_revenue": r["fcr_n_revenue"], "fcr_d_revenue": 0, "cycles": r["cycles"], "fcr_n_utilization": r["fcr_n_utilization"]})
    else: infeasible_counts["da_fcr_n"] += 1

    # DA + FCR ALL
    r = optimize_dispatch_multimarket(da_p, fcr_n, fcr_du, fcr_dd)
    if r.get("status") == "optimal":
        results["da_fcr_all"].append({"date": date, "revenue_net": r["revenue_net"], "da_revenue": r["da_revenue"], "fcr_n_revenue": r["fcr_n_revenue"], "fcr_d_revenue": r["fcr_d_revenue"], "cycles": r["cycles"], "fcr_n_utilization": r["fcr_n_utilization"]})
    else: infeasible_counts["da_fcr_all"] += 1

for name, rows in results.items():
    df_res = pd.DataFrame(rows)
    df_res.to_parquet(f"{PROCESSED_PATH}/se3_phase4_{name}.parquet")

print(f"\n✅ Backtest complete. Infeasible days: {infeasible_counts}")

Backtesting Days: 100%|██████████| 731/731 [02:04<00:00,  5.88it/s]



✅ Backtest complete. Infeasible days: {'da_only': 0, 'da_fcr_n': 2, 'da_fcr_all': 2}


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load results
da = pd.read_parquet(f"{PROCESSED_PATH}/se3_phase4_da_only.parquet")
da_fcr_n = pd.read_parquet(f"{PROCESSED_PATH}/se3_phase4_da_fcr_n.parquet")
da_fcr_all = pd.read_parquet(f"{PROCESSED_PATH}/se3_phase4_da_fcr_all.parquet")

# Summary stats
summary = []
for df_res, name in [(da, "DA Only"), (da_fcr_n, "DA + FCR-N"), (da_fcr_all, "DA + FCR-All")]:
    summary.append({
        "Scenario": name,
        "Total Revenue (€)": df_res["revenue_net"].sum(),
        "Avg Daily Rev (€)": df_res["revenue_net"].mean(),
        "Total Cycles": df_res["cycles"].sum(),
        "FCR-N Rev (€)": df_res["fcr_n_revenue"].sum(),
        "FCR-D Rev (€)": df_res["fcr_d_revenue"].sum()
    })

summary_df = pd.DataFrame(summary)
print("📊 Phase 4 Summary (2024–2025):")
print(summary_df.set_index("Scenario").round(0).to_string())

# Visualization
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Cumulative Net Revenue Comparison", "Monthly Revenue Composition (DA + FCR-All)"),
    vertical_spacing=0.15
)

# Panel 1: Cumulative
for df_res, name in [(da, "DA Only"), (da_fcr_n, "DA + FCR-N"), (da_fcr_all, "DA + FCR-All")]:
    fig.add_trace(go.Scatter(x=df_res["date"], y=df_res["revenue_net"].cumsum(), name=name), row=1, col=1)

# Panel 2: Composition for FCR-All
da_fcr_all['month'] = pd.to_datetime(da_fcr_all['date']).dt.to_period('M').astype(str)
monthly = da_fcr_all.groupby('month').agg({'da_revenue':'sum', 'fcr_n_revenue':'sum', 'fcr_d_revenue':'sum'}).reset_index()

fig.add_trace(go.Bar(x=monthly['month'], y=monthly['da_revenue'], name='DA Arbitrage', marker_color='cyan'), row=2, col=1)
fig.add_trace(go.Bar(x=monthly['month'], y=monthly['fcr_n_revenue'], name='FCR-N', marker_color='orange'), row=2, col=1)
fig.add_trace(go.Bar(x=monthly['month'], y=monthly['fcr_d_revenue'], name='FCR-D', marker_color='magenta'), row=2, col=1)

fig.update_layout(template="plotly_dark", height=800, barmode='stack', title="BESS Multi-Market Performance Analysis")
fig.show()

📊 Phase 4 Summary (2024–2025):
              Total Revenue (€)  Avg Daily Rev (€)  Total Cycles  FCR-N Rev (€)  FCR-D Rev (€)
Scenario                                                                                      
DA Only                 70369.0               97.0         681.0            0.0            0.0
DA + FCR-N             666525.0              917.0         210.0       635943.0            0.0
DA + FCR-All           752165.0             1035.0         248.0       523682.0       195343.0


In [ ]:
# Summary table
CAPEX = 560_000
OM    = 8_000
r     = 0.08
WARRANTY_CYCLES = 6_000
MAX_YEARS = 15

print(f"\n{'='*80}")
print(f"📊 PHASE 4 RESULTS — Revenue stacking in SE3 (2024–2025)")
print(f"{'='*80}")
print(f"\n{'Scenario':<22} {'DA rev':>9} {'FCR-N':>9} {'FCR-D':>9} {'Net':>10} {'Cycles':>8} {'Battery life':>13} {'NPV':>12}")
print("─" * 95)

phase4_summary = {}
for name, label in [("da_only", "DA only (baseline)"),
                     ("da_fcr_n", "DA + FCR-N"),
                     ("da_fcr_all", "DA + FCR-N + FCR-D")]:
    # Convert list of dicts to DataFrame for calculation
    df_r  = pd.DataFrame(results[name])
    yrs   = len(df_r) / 365.25

    ann_net   = df_r["revenue_net"].sum() / yrs
    ann_da    = df_r["da_revenue"].sum() / yrs
    ann_fcr_n = df_r["fcr_n_revenue"].sum() / yrs
    ann_fcr_d = df_r["fcr_d_revenue"].sum() / yrs
    ann_cyc   = df_r["cycles"].sum() / yrs

    life      = min(WARRANTY_CYCLES / ann_cyc if ann_cyc > 0 else 999, MAX_YEARS)
    cf        = ann_net - OM

    # Simple NPV calculation
    pv_cf     = cf * sum(1/(1+r)**t for t in range(1, int(life)+1))
    npv       = -CAPEX + pv_cf

    phase4_summary[name] = {"annual_net": ann_net, "npv": npv, "life": life}

    print(f"{label:<22} €{ann_da:>7,.0f} €{ann_fcr_n:>7,.0f} €{ann_fcr_d:>7,.0f} €{ann_net:>8,.0f} "
          f"{ann_cyc:>7.0f}    {life:>6.1f} yrs   €{npv:>10,.0f}")

# Key takeaway
da_npv   = phase4_summary["da_only"]["npv"]
all_npv  = phase4_summary["da_fcr_all"]["npv"]
da_rev   = phase4_summary["da_only"]["annual_net"]
all_rev  = phase4_summary["da_fcr_all"]["annual_net"]

print(f"\n{'='*80}")
print(f"💡 Revenue stacking uplift: DA only → DA+FCR-N+FCR-D")
print(f"   Revenue: €{da_rev:,.0f} → €{all_rev:,.0f}  (+€{all_rev-da_rev:,.0f}/year,  +{100*(all_rev-da_rev)/da_rev:.0f}%)")
print(f"   NPV:     €{da_npv:,.0f} → €{all_npv:,.0f}  (Δ €{all_npv-da_npv:,.0f})")

if all_npv >= 0:
    print(f"   ✅ NPV-POSITIVE: BESS is economically viable with full revenue stacking!")
else:
    annuity_factor = sum(1/(1+r)**t for t in range(1, 16))
    gap = -all_npv / annuity_factor
    print(f"   ⚠️  NPV still negative. Remaining gap: €{gap:,.0f}/MW/year more needed.")
    print(f"   → This gap could be closed by: intraday continuous, aFRR, or FFR (Phase 5 extension)")


📊 PHASE 4 RESULTS — Revenue stacking in SE3 (2024–2025)

Scenario                  DA rev     FCR-N     FCR-D        Net   Cycles  Battery life          NPV
───────────────────────────────────────────────────────────────────────────────────────────────
DA only (baseline)     € 46,581 €      0 €      0 €  35,257     341      15.0 yrs   €  -326,695
DA + FCR-N             € 22,468 €319,502 €      0 € 334,867     106      15.0 yrs   € 2,237,809
DA + FCR-N + FCR-D     € 24,503 €263,102 € 98,142 € 377,893     124      15.0 yrs   € 2,606,090

💡 Revenue stacking uplift: DA only → DA+FCR-N+FCR-D
   Revenue: €35,257 → €377,893  (+€342,636/year,  +972%)
   NPV:     €-326,695 → €2,606,090  (Δ €2,932,785)
   ✅ NPV-POSITIVE: BESS is economically viable with full revenue stacking!


In [ ]:
# Robustness check: cap FCR prices at 95th percentile
fcr_n_cap  = df_ph4["fcr_n"].quantile(0.95)
fcr_du_cap = df_ph4["fcr_d_up"].quantile(0.95)
fcr_dd_cap = df_ph4["fcr_d_dn"].quantile(0.95)
print(f"95th percentile caps:")
print(f"  FCR-N:    €{fcr_n_cap:.2f}/MW/h (vs mean €{df_ph4['fcr_n'].mean():.2f})")
print(f"  FCR-D up: €{fcr_du_cap:.2f}/MW/h")
print(f"  FCR-D dn: €{fcr_dd_cap:.2f}/MW/h")

df_ph4["fcr_n_capped"]    = df_ph4["fcr_n"].clip(upper=fcr_n_cap)
df_ph4["fcr_d_up_capped"] = df_ph4["fcr_d_up"].clip(upper=fcr_du_cap)
df_ph4["fcr_d_dn_capped"] = df_ph4["fcr_d_dn"].clip(upper=fcr_dd_cap)

95th percentile caps:
  FCR-N:    €84.28/MW/h (vs mean €37.28)
  FCR-D up: €24.44/MW/h
  FCR-D dn: €61.91/MW/h


In [ ]:
df_ph4[["timestamp","fcr_n","fcr_d_up","fcr_d_dn"]].to_parquet(
    f"{PROCESSED_PATH}/se3_fcr_prices_real_2024_2025.parquet"
)
print("✅ Real SVK FCR prices saved.")

✅ Real SVK FCR prices saved.


In [ ]:
from tqdm import tqdm

print("Running capped price robustness check...\n")

# Caps already computed
fcr_n_cap  = df_ph4["fcr_n"].quantile(0.95)
fcr_du_cap = df_ph4["fcr_d_up"].quantile(0.95)
fcr_dd_cap = df_ph4["fcr_d_dn"].quantile(0.95)

print(f"95th percentile caps: FCR-N €{fcr_n_cap:.2f} | FCR-D up €{fcr_du_cap:.2f} | FCR-D dn €{fcr_dd_cap:.2f}")
print(f"vs uncapped means:    FCR-N €{df_ph4['fcr_n'].mean():.2f} | FCR-D up €{df_ph4['fcr_d_up'].mean():.2f} | FCR-D dn €{df_ph4['fcr_d_dn'].mean():.2f}\n")

df_ph4["fcr_n_capped"]    = df_ph4["fcr_n"].clip(upper=fcr_n_cap)
df_ph4["fcr_d_up_capped"] = df_ph4["fcr_d_up"].clip(upper=fcr_du_cap)
df_ph4["fcr_d_dn_capped"] = df_ph4["fcr_d_dn"].clip(upper=fcr_dd_cap)

print(f"Capped means: FCR-N €{df_ph4['fcr_n_capped'].mean():.2f} | FCR-D up €{df_ph4['fcr_d_up_capped'].mean():.2f} | FCR-D dn €{df_ph4['fcr_d_dn_capped'].mean():.2f}")

# Run backtest with capped prices
capped_results = []
for date, group in tqdm(df_ph4.groupby("date"), desc="Capped backtest"):
    if len(group) < 23 or len(group) > 25:
        continue

    # Handle potential NaNs in price data before optimizing
    da_p   = group["price_eur_mwh"].fillna(0).values
    fcr_n  = group["fcr_n_capped"].fillna(0).values
    fcr_du = group["fcr_d_up_capped"].fillna(0).values
    fcr_dd = group["fcr_d_dn_capped"].fillna(0).values

    r = optimize_dispatch_multimarket(da_p, fcr_n, fcr_du, fcr_dd)

    if r and r.get("status") == "optimal":
        capped_results.append({
            "date": date,
            "revenue_net": r.get("revenue_net"),
            "da_revenue": r.get("da_revenue", 0),
            "fcr_n_revenue": r.get("fcr_n_revenue", 0),
            "fcr_d_revenue": r.get("fcr_d_revenue", 0),
            "cycles": r.get("cycles", 0),
        })

capped_df = pd.DataFrame(capped_results)
capped_df["date"] = pd.to_datetime(capped_df["date"])
capped_df.to_parquet(f"{PROCESSED_PATH}/se3_phase4_capped_results.parquet")

yrs = len(capped_df) / 365.25
ann_net   = capped_df["revenue_net"].sum() / yrs
ann_da    = capped_df["da_revenue"].sum() / yrs
ann_fcr_n = capped_df["fcr_n_revenue"].sum() / yrs
ann_fcr_d = capped_df["fcr_d_revenue"].sum() / yrs
ann_cyc   = capped_df["cycles"].sum() / yrs

CAPEX = 560_000; OM = 8_000; r_disc = 0.08; LIFE = 15
cf = ann_net - OM
pv = cf * sum(1/(1+r_disc)**t for t in range(1, LIFE+1))
npv = -CAPEX + pv

# Compare
orig_df = pd.read_parquet(f"{PROCESSED_PATH}/se3_phase4_da_fcr_all.parquet")
orig_ann = orig_df["revenue_net"].sum() / (len(orig_df)/365.25)

print(f"\n{'='*70}")
print(f"📊 ROBUSTNESS CHECK — Full-cap vs 95th-percentile-capped prices")
print(f"{'='*70}")
print(f"                     {'Uncapped (full SVK)':>22} {'95th pct capped':>20}")
print(f"Annual revenue:      €{orig_ann:>20,.0f} €{ann_net:>18,.0f}")
print(f"  of which DA:                            €{ann_da:>18,.0f}")
print(f"  of which FCR-N:                         €{ann_fcr_n:>18,.0f}")
print(f"  of which FCR-D:                         €{ann_fcr_d:>18,.0f}")
print(f"Annual cycles:                            {ann_cyc:>18.0f}")
print(f"NPV:                                      €{npv:>18,.0f}")

if npv >= 0:
    print(f"\n✅ NPV still positive after capping price spikes at 95th percentile")
    print(f"   Conclusion is ROBUST to extreme FCR price events")
else:
    print(f"\n⚠️  NPV turns negative after capping — result is spike-dependent")
print(f"\n💾 Capped results saved")

Running capped price robustness check...

95th percentile caps: FCR-N €84.28 | FCR-D up €24.44 | FCR-D dn €61.91
vs uncapped means:    FCR-N €37.28 | FCR-D up €8.28 | FCR-D dn €16.47

Capped means: FCR-N €33.95 | FCR-D up €7.50 | FCR-D dn €11.41


Capped backtest: 100%|██████████| 731/731 [00:48<00:00, 15.16it/s]



📊 ROBUSTNESS CHECK — Full-cap vs 95th-percentile-capped prices
                        Uncapped (full SVK)      95th pct capped
Annual revenue:      €             377,893 €           315,797
  of which DA:                            €            24,722
  of which FCR-N:                         €           254,167
  of which FCR-D:                         €            44,761
Annual cycles:                                           125
NPV:                                      €         2,074,582

✅ NPV still positive after capping price spikes at 95th percentile
   Conclusion is ROBUST to extreme FCR price events

💾 Capped results saved
